# Step 1

In [1]:
! pip install pandas numpy scikit-learn xgboost lightgbm matplotlib seaborn fastapi uvicorn django ollama


     ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
      --------------------------------------- 0.0/1.5 MB 660.6 kB/s eta 0:00:03
     - -------------------------------------- 0.1/1.5 MB 656.4 kB/s eta 0:00:03
     -- ------------------------------------- 0.1/1.5 MB 655.4 kB/s eta 0:00:03
     -- ------------------------------------- 0.1/1.5 MB 590.8 kB/s eta 0:00:03
     --- ------------------------------------ 0.1/1.5 MB 655.8 kB/s eta 0:00:02
     ---- ----------------------------------- 0.2/1.5 MB 655.4 kB/s eta 0:00:02
     ----- ---------------------------------- 0.2/1.5 MB 655.1 kB/s eta 0:00:02
     ------- -------------------------------- 0.3/1.5 MB 714.4 kB/s eta 0:00:02
     ------- -------------------------------- 0.3/1.5 MB 707.1 kB/s eta 0:00:02
     -------- ------------------------------- 0.3/1.5 MB 731.4 kB/s eta 0:00:02
     -------- ------------------------------- 0.3/1.5 MB 655.0 kB/s eta 0:00:02
     --------- ------------------------------ 0.3


[notice] A new release of pip is available: 23.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd

match_df = pd.read_csv('Match_Info.csv')
ball_df = pd.read_csv('Ball_By_Ball_Match_Data.csv')
players_df = pd.read_csv('2024_players_details.csv')
teams_df = pd.read_csv('teams_info.csv')

print(match_df.head())
print(ball_df.head())
print(players_df.head())
print(teams_df.head())


   match_number                        team1                        team2  \
0       1178403              Kings XI Punjab  Royal Challengers Bangalore   
1        980953          Sunrisers Hyderabad  Royal Challengers Bangalore   
2        419118             Rajasthan Royals        Kolkata Knight Riders   
3        548342             Delhi Daredevils               Mumbai Indians   
4       1082621  Royal Challengers Bangalore                Gujarat Lions   

   match_date                  toss_winner toss_decision result eliminator  \
0  2019-04-13  Royal Challengers Bangalore         field    Win        NaN   
1  2016-04-30  Royal Challengers Bangalore         field    Win        NaN   
2  2010-03-20             Rajasthan Royals           bat    Win        NaN   
3  2012-04-27               Mumbai Indians         field    Win        NaN   
4  2017-04-27                Gujarat Lions         field    Win        NaN   

                        winner  player_of_match  \
0  Royal Challeng

In [9]:
print(match_df.columns)


Index(['match_number', 'team1', 'team2', 'match_date', 'toss_winner',
       'toss_decision', 'result', 'eliminator', 'winner', 'player_of_match',
       'venue', 'city', 'team1_players', 'team2_players'],
      dtype='object')


In [10]:
def calculate_recent_form(df, team_col, result_col='winner'):
    team_form_dict = {}
    form_list = []
    for i, row in df.iterrows():
        team = row[team_col]
        past_results = team_form_dict.get(team, [])
        last_5 = past_results[-5:] if past_results else []
        win_ratio = sum(last_5) / 5 if last_5 else 0.5
        form_list.append(win_ratio)
        win = 1 if row[result_col] == team else 0
        team_form_dict.setdefault(team, []).append(win)
    return form_list

# ✅ Use correct column names
match_df['team1_form'] = calculate_recent_form(match_df, 'team1', result_col='winner')
match_df['team2_form'] = calculate_recent_form(match_df, 'team2', result_col='winner')


In [11]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Convert column names to match your dataset
# You already used lowercase column names like 'team1', 'winner', 'venue', etc.

# Encode categorical variables
match_df['toss_winner_code'] = match_df['toss_winner'].astype('category').cat.codes
match_df['venue_code'] = match_df['venue'].astype('category').cat.codes

# Rename the recent form columns if not already done
match_df.rename(columns={'team1_form': 'Team1_Form', 'team2_form': 'Team2_Form'}, inplace=True)

# Features and target
features = ['Team1_Form', 'Team2_Form', 'toss_winner_code', 'venue_code']
match_df = match_df.dropna(subset=features + ['winner', 'team1'])  # Drop any rows with missing values

X = match_df[features]
y = (match_df['winner'] == match_df['team1']).astype(int)  # 1 if team1 wins, else 0

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Model training
model = GradientBoostingClassifier()
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))


Accuracy: 0.45982142857142855


In [12]:
import pickle

with open('match_winner_model.pkl', 'wb') as f:
    pickle.dump(model, f)


In [15]:
# Track team form (already added earlier)
def calculate_recent_form(df, team_col, result_col='winner'):
    team_form_dict = {}
    form_list = []
    for _, row in df.iterrows():
        team = row[team_col]
        past_results = team_form_dict.get(team, [])
        last_5 = past_results[-5:] if past_results else []
        win_ratio = sum(last_5) / 5 if last_5 else 0.5
        form_list.append(win_ratio)
        win = 1 if row[result_col] == team else 0
        team_form_dict.setdefault(team, []).append(win)
    return form_list

# Create new columns
match_df = match_df.dropna(subset=['team1', 'team2', 'toss_winner', 'toss_decision', 'venue', 'winner'])
match_df['Team1_Form'] = calculate_recent_form(match_df, 'team1')
match_df['Team2_Form'] = calculate_recent_form(match_df, 'team2')


In [16]:
match_df['team1_code'] = match_df['team1'].astype('category').cat.codes
match_df['team2_code'] = match_df['team2'].astype('category').cat.codes
match_df['toss_winner_code'] = match_df['toss_winner'].astype('category').cat.codes
match_df['venue_code'] = match_df['venue'].astype('category').cat.codes
match_df['toss_decision_code'] = match_df['toss_decision'].map({'bat': 1, 'field': 0})
match_df['team1_won_toss'] = (match_df['team1'] == match_df['toss_winner']).astype(int)


In [17]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

features = ['Team1_Form', 'Team2_Form', 'team1_code', 'team2_code',
            'toss_winner_code', 'venue_code', 'toss_decision_code', 'team1_won_toss']
X = match_df[features]
y = (match_df['winner'] == match_df['team1']).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
gb_model = GradientBoostingClassifier(n_estimators=300, learning_rate=0.08, max_depth=5)
gb_model.fit(X_train, y_train)
print("Gradient Boosting Accuracy:", accuracy_score(y_test, gb_model.predict(X_test)))

Gradient Boosting Accuracy: 0.4732142857142857


In [18]:
# Hypothetical score values (you can extract from ball_df)
import numpy as np
match_df['winning_score'] = np.random.randint(140, 210, size=len(match_df))
match_df['losing_score'] = np.random.randint(90, 160, size=len(match_df))

from sklearn.ensemble import GradientBoostingRegressor
score_model = GradientBoostingRegressor()
score_model.fit(X_train, match_df.loc[X_train.index, 'winning_score'])

pred_score = score_model.predict(X_test)
print("Sample Winning Score Prediction:", pred_score[:5])


Sample Winning Score Prediction: [175.65260162 175.71325939 171.01340387 172.54699662 176.13135116]


In [23]:
print(ball_df.columns)


Index(['ID', 'Innings', 'Overs', 'BallNumber', 'Batter', 'Bowler',
       'NonStriker', 'ExtraType', 'BatsmanRun', 'ExtrasRun', 'TotalRun',
       'IsWicketDelivery', 'PlayerOut', 'Kind', 'FieldersInvolved',
       'BattingTeam'],
      dtype='object')


In [26]:
# Example player stats: Average runs/wickets per match
player_stats = ball_df.groupby('Batter').agg({'BatsmanRun': 'sum'}).reset_index()
player_stats.rename(columns={'BatsmanRun': 'Total_runs'}, inplace=True)
player_stats['matches_played'] = ball_df.groupby('Batter')['ID'].nunique().values
player_stats['avg_runs'] = player_stats['Total_runs'] / player_stats['matches_played']
print(player_stats.head())


           Batter  Total_runs  matches_played   avg_runs
0  A Ashish Reddy         280              23  12.173913
1        A Badoni         851              43  19.790698
2      A Chandila           4               2   2.000000
3        A Chopra          53               6   8.833333
4     A Choudhary          25               3   8.333333


In [29]:
# Assuming 'isWicketDelivery' column exists and is 1 when a wicket is taken
bowler_stats = ball_df.groupby('Bowler').agg({
    'IsWicketDelivery': 'sum',
    'TotalRun': 'sum',
    'ID': 'nunique'
}).reset_index()

bowler_stats.rename(columns={
    'IsWicketDelivery': 'total_wickets',
    'TotalRun': 'runs_conceded',
    'ID': 'matches_played'
}, inplace=True)

bowler_stats['avg_wickets'] = bowler_stats['total_wickets'] / bowler_stats['matches_played']
bowler_stats['economy'] = bowler_stats['runs_conceded'] / (bowler_stats['matches_played'] * 4)  # assuming 4 overs per match
print(bowler_stats.head())


           Bowler  total_wickets  runs_conceded  matches_played  avg_wickets  \
0  A Ashish Reddy             19            400              20     0.950000   
1        A Badoni              2             37               5     0.400000   
2      A Chandila             11            245              12     0.916667   
3     A Choudhary              5            144               5     1.000000   
4     A Dananjaya              0             47               1     0.000000   

     economy  
0   5.000000  
1   1.850000  
2   5.104167  
3   7.200000  
4  11.750000  


# Step 2

In [32]:
! pip install requests



[notice] A new release of pip is available: 23.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
set QROQ_API_KEY=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


SyntaxError: invalid syntax (3589164947.py, line 1)

In [37]:
! pip install transformers



[notice] A new release of pip is available: 23.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
from transformers import pipeline

# Load the model and tokenizer
generator = pipeline('text-generation', model='gpt2')

def query_huggingface(prompt):
    response = generator(prompt, max_length=100, num_return_sequences=1)
    return response[0]['generated_text']

# Example prompt
prompt = "Explain why team1 has a better chance to win against team2 based on recent form."
print(query_huggingface(prompt))


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

c:\Users\sande\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sande\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Explain why team1 has a better chance to win against team2 based on recent form.


Reply ~13000 0 ~20 min 7 By : jd_plasma : The best build against team1.


Reply ~11000 0 ~40 min 2 By : rofandro : The best build for the mid game for team1.


Reply ~37000 0 ~40 min 5 By : S.Lamp : the best item to buy from Team 1


In [39]:
# Step 1: Get the prediction from your GradientBoosting model or other models
team1_prediction = "Team1"  # Example of the model's prediction
team2_prediction = "Team2"  # Example of the model's prediction

# Step 2: Define dynamic prompts based on predictions
prompt_match_winner = f"Why is {team1_prediction} predicted to win against {team2_prediction} based on recent team performance, venue, and other factors?"

# Step 3: Query Hugging Face model
response_match_winner = query_huggingface(prompt_match_winner)
print(response_match_winner)

# Example for player performance prediction
key_player_prediction = "Player1 is predicted to score 50 runs."  # Replace with actual prediction
prompt_player_performance = f"Why is {key_player_prediction} likely to happen? Explain based on recent performance trends."

response_player_performance = query_huggingface(prompt_player_performance)
print(response_player_performance)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Why is Team1 predicted to win against Team2 based on recent team performance, venue, and other factors?

We look at all the data we've collected to come to a conclusion. We don't do a whole lot with just one set of results. But we also look at whether the team is at least the winning team. Theoretically, this might not matter as much as when you're looking at each individual performance, but it's interesting, since many people might be doing an
Why is Player1 is predicted to score 50 runs. likely to happen? Explain based on recent performance trends. If there's a reason to be concerned, that won't be the issue.

So why bother?

There are a lot more people out there who don't understand the problem with this model than just getting the ball into the glove and waiting for it to rain. That sucks. That's where you have to be. And that can only happen in the middle of the park


In [40]:
def generate_match_winner_explanation(team1_prediction, team2_prediction):
    prompt = f"Why is {team1_prediction} predicted to win against {team2_prediction} based on recent team form, venue, weather, and other match-related factors?"
    return query_huggingface(prompt)

def generate_player_performance_explanation(player_name, prediction):
    prompt = f"Why is {player_name} predicted to perform well in this match, with an expected performance of {prediction}? Provide insights based on recent form and match conditions."
    return query_huggingface(prompt)

# Example usage
team1_prediction = "Team1"
team2_prediction = "Team2"
player_name = "Player1"
player_prediction = "50 runs"

match_explanation = generate_match_winner_explanation(team1_prediction, team2_prediction)
print(match_explanation)

player_explanation = generate_player_performance_explanation(player_name, player_prediction)
print(player_explanation)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Why is Team1 predicted to win against Team2 based on recent team form, venue, weather, and other match-related factors?

In any way they would win against Team2 in the finals of the playoffs.

How can I predict against Team1 based on their performance against the European teams during the recent playoffs?

For example: if the European teams have 2 losses against Team1, which you think will be Team1's 2nd loss of the entire year, should
Why is Player1 predicted to perform well in this match, with an expected performance of 50 runs? Provide insights based on recent form and match conditions.

3. How likely is Player1 to score from within the three set play phases (30s, 60s, or 80s)?

Players always have a chance of scoring from within the three set play phases (30s, 60s, or 80S) when playing with a team which has three set play phases where there are


In [41]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class MatchInput(BaseModel):
    team1: str
    team2: str
    recent_form_team1: float
    recent_form_team2: float
    venue: str
    weather: str

@app.post("/predict_match_winner")
def predict_match_winner(input_data: MatchInput):
    # Run your ensemble model to predict match winner based on input data
    # Example placeholder prediction
    winner = "Team1"  # Replace with actual model prediction
    explanation = generate_match_winner_explanation(input_data.team1, input_data.team2)
    return {"prediction": winner, "explanation": explanation}

@app.post("/explain_player_performance")
def explain_player_performance(player_name: str, predicted_performance: str):
    explanation = generate_player_performance_explanation(player_name, predicted_performance)
    return {"explanation": explanation}


In [43]:
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import pipeline

# Initialize the Hugging Face pipeline for text generation
generator = pipeline('text-generation', model='gpt2')

# Initialize FastAPI app
app = FastAPI()

# Define Pydantic models for the inputs
class MatchInput(BaseModel):
    team1: str
    team2: str
    team1_recent_form: float
    team2_recent_form: float
    venue: str
    weather: str

class PlayerInput(BaseModel):
    player_name: str
    predicted_performance: str

# Function to query Hugging Face for text generation
def query_huggingface(prompt):
    response = generator(prompt, max_length=100, num_return_sequences=1)
    return response[0]['generated_text']

# Function to generate explanation for match winner prediction
def generate_match_winner_explanation(team1, team2):
    prompt = f"Why is {team1} predicted to win against {team2} based on recent form, venue, and weather conditions?"
    return query_huggingface(prompt)

# Function to generate explanation for player performance
def generate_player_performance_explanation(player_name, predicted_performance):
    prompt = f"Why is {player_name} predicted to perform well in this match, with an expected performance of {predicted_performance}? Explain based on recent form and conditions."
    return query_huggingface(prompt)

# Endpoint to predict match winner and explain
@app.post("/predict_match_winner")
def predict_match_winner(input_data: MatchInput):
    # Placeholder for actual model prediction
    predicted_winner = "Team1"  # Replace this with actual model prediction logic
    explanation = generate_match_winner_explanation(input_data.team1, input_data.team2)
    return {"prediction": predicted_winner, "explanation": explanation}

# Endpoint to explain player performance
@app.post("/explain_player_performance")
def explain_player_performance(input_data: PlayerInput):
    explanation = generate_player_performance_explanation(input_data.player_name, input_data.predicted_performance)
    return {"explanation": explanation}



Device set to use cpu


In [48]:
! pip install fastapi uvicorn nest_asyncio



[notice] A new release of pip is available: 23.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [50]:
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from transformers import pipeline
import uvicorn
import subprocess

# Initialize the Hugging Face pipeline for text generation
generator = pipeline('text-generation', model='gpt2')

# Initialize FastAPI app
app = FastAPI()

# Define Pydantic models for the inputs
class MatchInput(BaseModel):
    team1: str
    team2: str
    team1_recent_form: float
    team2_recent_form: float
    venue: str
    weather: str

class PlayerInput(BaseModel):
    player_name: str
    predicted_performance: str

# Function to query Hugging Face for text generation
def query_huggingface(prompt):
    response = generator(prompt, max_length=100, num_return_sequences=1)
    return response[0]['generated_text']

# Function to generate explanation for match winner prediction
def generate_match_winner_explanation(team1, team2):
    prompt = f"Why is {team1} predicted to win against {team2} based on recent form, venue, and weather conditions?"
    return query_huggingface(prompt)

# Function to generate explanation for player performance
def generate_player_performance_explanation(player_name, predicted_performance):
    prompt = f"Why is {player_name} predicted to perform well in this match, with an expected performance of {predicted_performance}? Explain based on recent form and conditions."
    return query_huggingface(prompt)

# Endpoint to predict match winner and explain
@app.post("/predict_match_winner")
def predict_match_winner(input_data: MatchInput):
    # Placeholder for actual model prediction
    predicted_winner = "Team1"  # Replace this with actual model prediction logic
    explanation = generate_match_winner_explanation(input_data.team1, input_data.team2)
    return {"prediction": predicted_winner, "explanation": explanation}

# Endpoint to explain player performance
@app.post("/explain_player_performance")
def explain_player_performance(input_data: PlayerInput):
    explanation = generate_player_performance_explanation(input_data.player_name, input_data.predicted_performance)
    return {"explanation": explanation}

# Run the FastAPI app in Jupyter using uvicorn
nest_asyncio.apply()

# Run uvicorn in a subprocess
subprocess.Popen(["uvicorn", "filename:app", "--host", "0.0.0.0", "--port", "8000"])


Device set to use cpu


<Popen: returncode: None args: ['uvicorn', 'filename:app', '--host', '0.0.0....>